In [1]:
# Ep5 Code Lab: Free APIs & Integration
# Google Colab version

!pip -q install requests

import requests
import json
from google.colab import userdata

# 1. CONFIG
BASE_URL = "https://api.groq.com/openai/v1/chat/completions"
MODEL_NAME = "openai/gpt-oss-120b"

# Read API key from Colab Secrets
# In Colab: click the key icon (Secrets) and create GROQ_API_KEY
API_KEY = userdata.get("GROQ_API_KEY")

if not API_KEY:
    raise ValueError("GROQ_API_KEY not found in Colab Secrets.")

print("Connected. Ready.")

# 2. QUICK TEST — raw request
def raw_call(prompt, model=MODEL_NAME):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }

    response = requests.post(BASE_URL, headers=headers, json=payload)
    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"]

print(raw_call("Say hello in one short sentence."))

Connected. Ready.
Hello!


In [2]:
# 3. THE 5 UNIVERSAL API COMPONENTS

# 1. BASE URL — address of the model server
BASE_URL = "https://api.groq.com/openai/v1/chat/completions"

# 2. AUTH KEY — proves you are authorised
API_KEY = userdata.get("GROQ_API_KEY")

# 3. AUTH HEADER
headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

# 4. MODEL STRING + 5. MESSAGES LIST
payload = {
    "model": "openai/gpt-oss-120b",
    "messages": [
        {"role": "user", "content": "Hello"}
    ]
}

# Send and parse
response = requests.post(BASE_URL, headers=headers, json=payload)
response.raise_for_status()
data = response.json()

print(data["choices"][0]["message"]["content"])

Hello! How can I help you today?


In [3]:
# 4. THE MESSAGES LIST IN DEPTH

full_call = requests.post(
    BASE_URL,
    headers=headers,
    json={
        "model": "openai/gpt-oss-120b",
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is the capital of France?"}
        ]
    }
)

full_call.raise_for_status()
data = full_call.json()

print("Answer:", data["choices"][0]["message"]["content"])
print("Tokens used:", data.get("usage", {}).get("total_tokens", "Not returned"))

Answer: The capital of France is **Paris**.
Tokens used: 115


In [4]:
# 5. WRAPPER CLASS

class GroqClient:
    def __init__(self, api_key, base_url=BASE_URL, default_model=MODEL_NAME):
        self.api_key = api_key
        self.base_url = base_url
        self.default_model = default_model

    def chat(self, messages, model=None, temperature=0):
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

        payload = {
            "model": model or self.default_model,
            "messages": messages,
            "temperature": temperature
        }

        response = requests.post(self.base_url, headers=headers, json=payload)
        response.raise_for_status()
        data = response.json()

        return data["choices"][0]["message"]["content"]

client = GroqClient(api_key=API_KEY)

In [5]:
# 6. TESTING THE WRAPPER

# Single call
answer = client.chat([
    {"role": "user", "content": "What is 2 + 2?"}
])
print("Single call:", answer)

# Multi-turn conversation
conversation = [
    {"role": "user", "content": "My name is Alex."},
    {"role": "assistant", "content": "Hello Alex!"},
    {"role": "user", "content": "What is my name?"}
]

print()
print("Multi-turn:", client.chat(conversation))

Single call: 2 + 2 = 4.

Multi-turn: Your name is Alex.


In [6]:
# 7. FIVE EPISODES INTO ONE PIPELINE

user_name = "Alex"

system_instruction = f"""
ROLE: You are a digital twin for {user_name}.
KNOWLEDGE: Activity data from Ep1, embeddings from Ep2, attention from Ep3, prompting strategy from Ep4.
CONSTRAINTS: Only recommend based on user data.
TONE: Direct and analytical.
"""

twin_messages = [
    {"role": "system", "content": system_instruction},
    {"role": "user", "content": "What should I do next?"}
]

response = client.chat(twin_messages)
print(response)

**Next Action Recommendation (based on your recent activity patterns, embedding signals, and attention focus):**

1. **Finalize the current deliverable**  
   - You’ve spent the last 3 days deep‑diving into the data set (Ep1 activity) and your embeddings (Ep2) show a strong semantic shift toward “conclusion synthesis.”  
   - Allocate a focused 90‑minute block to write the **Results & Discussion** section, using the outline you drafted yesterday.

2. **Validate key assumptions**  
   - Your attention heatmap (Ep3) indicates lingering uncertainty around the “model validation” metric.  
   - Run the quick validation script you saved in the “scripts/validation” folder and record the output. If the confidence interval exceeds ±5 %, flag it for a follow‑up meeting.

3. **Schedule a brief sync**  
   - Your prompting strategy (Ep4) suggests you benefit from external feedback after each major writing chunk.  
   - Book a 15‑minute check‑in with Jordan by 11 AM tomorrow to review the draft you

In [8]:
# 8. STRETCH: TWO-TURN CONVERSATION

two_turn_messages = [
    {"role": "system", "content": system_instruction},
    {"role": "user", "content": "What should I do next?"},
    {"role": "assistant", "content": client.chat(twin_messages)},
    {"role": "user", "content": "Give me the top 3 priorities only."}
]

print(client.chat(two_turn_messages))

**Top 3 Priorities (ordered by urgency and current attention alignment)**  

1. **Task A – Deadline tomorrow** – Highest urgency; your attention heatmap shows sustained focus on its “data‑analysis” component.  
2. **Task B – Due in 3 days (medium priority)** – Next‑closest deadline; embeddings indicate you’re still in a “project‑planning” mindset that suits this task.  
3. **Task C – Due next week (low priority)** – Lowest urgency; keep it on the backlog until Tasks A and B are completed.
